# PBIT Safety Baseline — Classifier Comparison

**Perspective**: safety (severity classification).  
**Primary metrics**: CMR, Downward Risk, Severity Error Rate.  
**Secondary metrics**: Balanced Accuracy, Macro F1, F06 recall.

Grouped five-fold CV (StratifiedGroupKFold by Serial_Number).  
Augmentation: F04/F05/F06 → target = 160, noise = 1% (project setting).  
Models: LightGBM and XGBoost, default hyperparameters, all 6 weighting configs.

Primary sort key: **CMR** (lower = safer).

## Setup

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks_safety':
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

import lightgbm as lgb

from pbit.modeling import (
    META_COLS, FAULT_ORDER, EXPERIMENTS,
    XGBWrapper,
    make_cv_folds, run_cv_experiment, summarize_cv,
    safety_results_table,
)
from pbit.data_validation import load_long_dataset

TRAIN_PATH   = PROJECT_ROOT / 'data' / 'train_real.csv'
LONG_PATH    = PROJECT_ROOT / 'data' / 'PBIT_atuador_DESCARAC_r1_long_clean.csv'
RESULTS_PATH = PROJECT_ROOT / 'data' / 'safety_baseline_results.csv'

AUG_TARGET = 160

PROJECT_ROOT

## Load Data & Folds

In [ ]:
train_df  = pd.read_csv(TRAIN_PATH)
long_df   = load_long_dataset(LONG_PATH)
feat_cols = [c for c in train_df.columns if c not in META_COLS]
folds     = make_cv_folds(train_df, n_splits=5, random_state=42)

print(f'Training set : {len(train_df):,} experiments x {len(feat_cols)} features')
print(f'Folds        : {len(folds)}')
print()
print('Train class distribution:')
print(train_df['Fault_Mode'].value_counts().reindex(FAULT_ORDER).to_string())

---
## LightGBM — all 6 configs

In [ ]:
lgb_params = dict(n_estimators=100, random_state=42, verbose=-1)

lgb_summaries = {}
t0 = time.time()
for cfg in EXPERIMENTS:
    fold_res = run_cv_experiment(
        lgb.LGBMClassifier(**lgb_params), cfg,
        train_df, long_df, feat_cols, folds,
        aug_target=AUG_TARGET,
    )
    s = summarize_cv(fold_res)
    label = f'LightGBM / {cfg.name}'
    lgb_summaries[label] = s
    print(f'{label:<35}  CMR={s["cmr_mean"]:.3f}  DR={s["down_risk_mean"]:.3f}  '
          f'SevErr={s["sev_err_mean"]:.3f}  F1={s["macro_f1_mean"]:.3f}  F06={s["f06_recall_mean"]:.3f}')

print(f'\nDone in {time.time()-t0:.0f}s')

## XGBoost — all 6 configs

In [ ]:
xgb_params = dict(n_estimators=100, eval_metric='mlogloss', random_state=42, verbosity=0)

xgb_summaries = {}
t0 = time.time()
for cfg in EXPERIMENTS:
    fold_res = run_cv_experiment(
        XGBWrapper(**xgb_params), cfg,
        train_df, long_df, feat_cols, folds,
        aug_target=AUG_TARGET,
    )
    s = summarize_cv(fold_res)
    label = f'XGBoost / {cfg.name}'
    xgb_summaries[label] = s
    print(f'{label:<35}  CMR={s["cmr_mean"]:.3f}  DR={s["down_risk_mean"]:.3f}  '
          f'SevErr={s["sev_err_mean"]:.3f}  F1={s["macro_f1_mean"]:.3f}  F06={s["f06_recall_mean"]:.3f}')

print(f'\nDone in {time.time()-t0:.0f}s')

---
## Safety Results Table

Sorted by CMR (ascending). Columns follow the paper's Table 4 order.

In [ ]:
all_summaries = {**lgb_summaries, **xgb_summaries}
tbl = safety_results_table(all_summaries)
tbl

## Safety Profile — LightGBM vs XGBoost

In [ ]:
cfg_names = [c.name for c in EXPERIMENTS]
x = np.arange(len(cfg_names))
w = 0.35

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, metric, ylabel in zip(
    axes,
    ['cmr_mean', 'down_risk_mean', 'sev_err_mean'],
    ['CMR', 'Downward Risk', 'Severity Error Rate'],
):
    lgb_vals = [lgb_summaries.get(f'LightGBM / {c}', {}).get(metric, float('nan')) for c in cfg_names]
    xgb_vals = [xgb_summaries.get(f'XGBoost / {c}',  {}).get(metric, float('nan')) for c in cfg_names]
    ax.bar(x - w/2, lgb_vals, w, label='LightGBM', color='steelblue',  alpha=0.85)
    ax.bar(x + w/2, xgb_vals, w, label='XGBoost',  color='darkorange', alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(cfg_names, rotation=30, ha='right', fontsize=8)
    ax.set_title(ylabel)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=8)
    ax.grid(True, axis='y', linewidth=0.3)

fig.suptitle(f'Safety baseline — LightGBM vs XGBoost (aug target={AUG_TARGET})', fontsize=12)
plt.tight_layout()
plt.show()

## CMR Heatmap — per config and model

In [ ]:
heatmap_data = pd.DataFrame({
    'LightGBM': [lgb_summaries.get(f'LightGBM / {c}', {}).get('cmr_mean', float('nan')) for c in cfg_names],
    'XGBoost':  [xgb_summaries.get(f'XGBoost / {c}',  {}).get('cmr_mean', float('nan')) for c in cfg_names],
}, index=cfg_names)

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn_r',
            vmin=0, vmax=0.15, linewidths=0.4, ax=ax)
ax.set_title(f'CMR per config (lower = safer, aug target={AUG_TARGET})')
plt.tight_layout()
plt.show()

## Save Results

In [ ]:
tbl.to_csv(RESULTS_PATH, index=False)
print('Saved:', RESULTS_PATH)
tbl